# Partie 2 - Traitement batch avec SparkCore (RDD)
Prerequis (depuis la machine hote) :
```bash
docker compose exec jupyter python3 producer/producer.py --mode file --n 50000
./scripts/upload_to_lake.sh        # copie transactions_history.jsonl -> lake/bronze/history/
```

In [ ]:
import os, sys, json
sys.path.insert(0, "/home/sahlpay/jobs"); sys.path.insert(0, "../jobs")
from sahlpay_common import get_spark, Paths, LOG_DIR, merchants_path, timer

spark = get_spark("SahlPay-P2-RDD", cores_max=4)
sc = spark.sparkContext
timings = {}
print(spark.version, sc.master, "defaultParallelism =", sc.defaultParallelism)

## 1. Chargement en RDD et parsing JSON

In [ ]:
history_file = f"{Paths.bronze_history}/transactions_history.jsonl"
lines = sc.textFile(history_file)                     # transformation (lazy)
events = lines.map(json.loads)                        # transformation (lazy)
events.cache()                                        # reutilise par plusieurs actions
with timer("p2_count_events", timings):
    n_events = events.count()                         # action -> declenche la lecture
print("Nombre d'evenements :", n_events, "| partitions :", events.getNumPartitions())
print(events.first())

## 2. Filtrer les transactions COMPLETED
Le flux contient ~2 % de doublons (meme `tx_id` et meme `status`). Un `COMPLETED` est unique par
transaction : on dedoublonne donc par `tx_id` pour ne pas compter deux fois un encaissement.

In [ ]:
completed_raw = events.filter(lambda e: e["status"] == "COMPLETED")
completed = (completed_raw
             .map(lambda e: (e["tx_id"], e))
             .reduceByKey(lambda a, b: a)              # garde un seul evenement par tx_id
             .values())
completed.cache()
with timer("p2_filter_completed", timings):
    n_raw, n_dedup = completed_raw.count(), completed.count()
print(f"COMPLETED bruts = {n_raw} | apres dedoublonnage = {n_dedup} | doublons retires = {n_raw - n_dedup}")

## 3. Montant total encaisse par ville (`map` + `reduceByKey`)

In [ ]:
with timer("p2_total_by_city", timings):
    total_by_city = (completed
                     .map(lambda e: (e["city"], e["amount_mad"]))
                     .reduceByKey(lambda a, b: a + b))
    total_list = total_by_city.collect()
for city, total in sorted(total_list, key=lambda x: -x[1]):
    print(f"{city:<12} {total:>15,.2f} MAD")

## 4. Montant moyen par transaction et par ville (somme et nombre en un seul passage)
`aggregateByKey(zero, seqOp, combOp)` : agregation locale dans chaque partition (`seqOp`), puis
fusion des resultats partiels apres le shuffle (`combOp`).

In [ ]:
with timer("p2_avg_by_city", timings):
    sum_count = (completed
                 .map(lambda e: (e["city"], e["amount_mad"]))
                 .aggregateByKey((0.0, 0),
                                 lambda acc, v: (acc[0] + v, acc[1] + 1),
                                 lambda a, b: (a[0] + b[0], a[1] + b[1])))
    city_stats = sum_count.mapValues(lambda sc_: (sc_[0], sc_[1], sc_[0] / sc_[1])).collect()

print(f"{'Ville':<12} {'Montant total (MAD)':>20} {'Nb tx':>8} {'Montant moyen (MAD)':>20}")
for city, (s, n, avg) in sorted(city_stats, key=lambda x: -x[1][0]):
    print(f"{city:<12} {s:>20,.2f} {n:>8} {avg:>20,.2f}")

## 5. Top 5 des commercants par nombre de paiements valides

In [ ]:
merchant_names = (sc.textFile(merchants_path()).map(json.loads)
                    .map(lambda m: (m["merchant_id"], (m["name"], m["city"], m["category"])))
                    .collectAsMap())
bnames = sc.broadcast(merchant_names)

with timer("p2_top5_merchants", timings):
    top5 = (completed
            .filter(lambda e: e["type"] == "P2M")
            .map(lambda e: (e["payee_id"], 1))
            .reduceByKey(lambda a, b: a + b)
            .takeOrdered(5, key=lambda kv: -kv[1]))
for rank, (mid, n) in enumerate(top5, 1):
    name, city, cat = bnames.value.get(mid, ("?", "?", "?"))
    print(f"{rank}. {mid} {name:<28} {city:<11} {cat:<12} {n} paiements")

## 6. Lineage (`toDebugString`) - transformations vs actions

In [ ]:
print(total_by_city.toDebugString().decode())

| Operation utilisee | Nature |
|---|---|
| `textFile`, `map`, `filter`, `reduceByKey`, `aggregateByKey`, `mapValues`, `values` | **transformations** (lazy : construisent le lineage, aucun calcul) |
| `cache` | marque le RDD pour la mise en memoire (lazy, effective a la 1re action) |
| `count`, `first`, `collect`, `collectAsMap`, `takeOrdered`, `take` | **actions** (declenchent un job et renvoient un resultat au driver) |

Chaque indentation `+-` du `toDebugString` marque une frontiere de **shuffle** (nouveau stage) :
ici `reduceByKey(tx_id)` (dedoublonnage) puis `reduceByKey(city)`.

## 7. DAG, stages et shuffle (a capturer dans Spark UI : http://localhost:4040 > Jobs)
On relance un job "propre" (sans cache) pour obtenir un DAG lisible a capturer.

In [ ]:
sc.setJobDescription("P2 - DAG montant total par ville (capture rapport)")
with timer("p2_dag_job_nocache", timings):
    dag_job = (sc.textFile(history_file).map(json.loads)
               .filter(lambda e: e["status"] == "COMPLETED")
               .map(lambda e: (e["city"], e["amount_mad"]))
               .reduceByKey(lambda a, b: a + b))
    dag_job.collect()
sc.setJobDescription(None)
print(dag_job.toDebugString().decode())

**A commenter dans le rapport**
* Ce job a **2 stages** : Stage 0 = `textFile -> map(json.loads) -> filter -> map` + ecriture
  du *shuffle write* (pre-agregation locale par ville grace au combiner de `reduceByKey`) ;
  Stage 1 = *shuffle read* + `reduceByKey` final + `collect`. Les operations "etroites"
  (narrow : `map`, `filter`) sont pipelinees dans le meme stage.
* `reduceByKey` provoque un **shuffle** : toutes les paires d'une meme ville doivent se retrouver
  dans la meme partition. Les donnees sont partitionnees par hash de la cle, ecrites sur disque
  local par les executors du stage 0, puis lues a travers le reseau par le stage 1. Grace a la
  pre-agregation cote map, on n'echange qu'une ligne par (partition, ville).
* **Tolerance aux pannes par le lineage** : un RDD ne stocke pas ses donnees de maniere
  repliquee ; il memorise la suite de transformations qui permet de le recalculer. Si un executor
  (ex. `spark-worker-2`) disparait, Spark recalcule seulement les partitions perdues a partir de la
  source (fichier dans MinIO) en rejouant `textFile -> map -> filter -> map`. Les fichiers de
  shuffle perdus entrainent la re-execution des tasks du stage parent correspondant.

In [ ]:
os.makedirs(LOG_DIR, exist_ok=True)
with open(os.path.join(LOG_DIR, "timings_part2_rdd.json"), "w") as f:
    json.dump(timings, f, indent=2)
print(timings)

In [ ]:
spark.stop()